In [45]:
import newkernelo as lib
import numpy as np
import time
import os.path
import json

In [46]:
print(lib.__doc__)

model = lib.TestModel()

# print(model.__doc__)
# print(model.get_D_dimension())
# print(model.F.__doc__)
# print(model.get_L_dimension())
# help(model.F)

# print(lib.TestModel.__doc__)




        Kernelo
        -----------------------
        Functional
        Learning
        DataGeneration
        ...
    


In [47]:
e = np.exp(1)
x = np.ones(4)

# y = np.zeros(9)
# t0 = time.time()
# for i in range(10000000): #10000000
#     model.F(x,y)
# tf = time.time() - t0
# print("Time F by reference: {}".format(tf))

# t0 = time.time()
# for i in range(10000000):
#     z = model.F(x)
# tf = time.time() - t0
# print("Time F by value: {}".format(tf))

# true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
# print("true_result")
# print(true_result)

z = model.F(x)
print(z)
print(x)
print(x.shape)
model.to_physic(x)
print(x)
print(x.shape)
w = model.to_physic(x)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
model.from_physic(x)
print(x)
y = model.from_physic(x)
print(x)
print(y)
z = model.from_physic(y)
print(x)
print(y)
print(z)

[[ 8.15484549  0.67957046  1.35914091  4.07742274  0.27182818 -0.67957046
  -1.6309691   1.35914091 -0.95139864]]
[1. 1. 1. 1.]
(4,)
[1. 1. 1. 1.]
(4,)
[1. 1. 1. 1.]
[[2. 2. 2. 2.]]
(1, 4)
=========== From physic ==========
[1. 1. 1. 1.]
[1. 1. 1. 1.]
[1. 1. 1. 1.]
[[0.5 0.5 0.5 0.5]]
[1. 1. 1. 1.]
[[0.5 0.5 0.5 0.5]]
[[0.25 0.25 0.25 0.25]]
HEYYY
   1.0000   1.0000   1.0000   1.0000



In [51]:
print(os.getcwd())
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = 3
col_size = D

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]


# # Create JSON file with geometries
geometries = np.empty((row_size,col_size))
var_geom = ["inc", "eme", "phi"]
for j in range(3):
    i=0
    for v in data[var_geom[j]]:
        geometries[j,i] = v
        i+=1

# geom = {'geometries': [[geometries[j,:].tolist()] for j in range(3)]
# }
# with open('geometries_shkuratov.json', 'w') as fp:
#     json.dump(geom, fp)

## INTEGRATION au code C++
variant = "5p"
physicalModel = lib.ShkuratovModel(geometries, variant, scalingCoeffs, offset)


### TEST
N = 10000
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    n=0
    for v in data[variables[l]]:
        photometries[l,n] = (float(v) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for v in data["y"]:
    l=0
    for w in v:
        expected_results[l,n] = float(w)
        l+=1
    n+=1


# compute results from the model
result = np.empty((D,))
assert_list = []
# for n in range(N):
#     result = physicalModel.F(photometries[:,n])
#     assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

photometries_vec = photometries[:,1]
print(photometries)
print(photometries.shape)
print(photometries_vec)
print(photometries_vec.shape)
test = np.array(photometries_vec) # ça marche
result = physicalModel.F(test)
assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))


# print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])

/home/luc/Documents/dev/kernelo-gllim-is/new_kernelo/python
[[0.08425903 0.7679216  0.30756917 ... 0.56352979 0.27240845 0.53810004]
 [0.91973274 0.83021563 0.66670211 ... 0.84324373 0.72732898 0.11056148]
 [0.18074199 0.47533843 0.24223638 ... 0.29665432 0.17614758 0.16531243]
 [0.05883782 0.07465033 0.66483779 ... 0.04891415 0.0529227  0.16488135]
 [0.66702887 0.9790033  0.60810938 ... 0.34188583 0.13040182 0.0897537 ]]
(5, 10000)
[0.7679216  0.83021563 0.47533843 0.07465033 0.9790033 ]
(5,)
HEYYY
   0.7679   0.8302   0.4753   0.0747   0.9790



RuntimeError: element-wise multiplication: incompatible matrix dimensions: 1x5 and 5x1